# Kernel 2: Tanh

The goal for this assignment is to implement the [Hyperbolic Tangent function](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Tanh.html).

<div style="text-align: center;">
    <img src="assets/tanh.png" width="500">
</div>

**Input & Output**: 1 dimensional tensor of $N$ float32 values.

In [2]:
import torch
import torch.nn.functional as F


N = 1024
x = torch.randn(N, dtype=torch.float32, device="cuda")
out = torch.empty_like(x)
print(x.shape)

torch.Size([1024])


### Assignment

In [ ]:
import flydsl.compiler as flyc
import flydsl.expr as fx


def ceildiv(x: int, y: int) -> int:
    return (x + y - 1) // y


@flyc.kernel
def tanh_kernel(x: fx.Pointer, out: fx.Pointer, n: fx.Int32):
    # TODO: write your kernel code here
    pass

@flyc.jit
def my_tanh(x: fx.Pointer, out: fx.Pointer, n: fx.Int32, stream: fx.Stream):
    # TODO: write your launch code here
    pass

Once your code is ready use this code to verify its correctness

In [6]:
out.zero_()

my_tanh(x, out, N, torch.cuda.default_stream())

ok = torch.allclose(out, F.tanh(x), rtol=1e-5, atol=1e-5)
if ok:
    print('Tanh kernel is correct!')
else:
    print('Nope...')

# Expected output: 'Tanh kernel is correct!'

Tanh kernel is correct!


## Solution


In [ ]:

@flyc.kernel
def tanh_kernel(x: fx.Pointer, out: fx.Pointer, n: fx.Int32):
    tid = fx.block_idx.x * fx.block_dim.x + fx.thread_idx.x
    
    if tid >= n:
        return
    
    x_i = x[tid]
    e = fx.exp(x_i)
    en = fx.exp(-x_i)
    out[tid] = (e - en) / (e + en)

@flyc.jit
def my_tanh(x: fx.Pointer, out: fx.Pointer, n: fx.Int32, stream: fx.Stream):
    block_dim = 128
    grid_x = ceildiv(n, block_dim)
    tanh_kernel(x, out, n).launch(grid=(grid_x, 1, 1), block=(block_dim, 1, 1), stream=stream)